# 03 — Term Structure QA and v0.7.1 Repair

This notebook reviews and repairs the v0.7 SPX VIX-style term-structure history.

**Inputs**
- Raw v0.7 term-structure file from the production build notebook.
- Cboe VIX and VIX9D histories from the benchmark notebook.

**Outputs**
- Final repaired research file: `vix_term_structure_history_v0_7_1_repaired_total_variance.parquet`
- Matching CSV copy
- Repair log and QA audit files

**Method**
- Keep raw v0.7 unchanged.
- Repair only dates with clear internal term-structure problems.
- Repair in **total variance space**, not annualized vol space.
- Use neighboring clean curves plus Cboe VIX9D / VIX anchors.
- Save full audit logs.

## 1. Project paths and configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# If this notebook is inside C:\Users\patri\vrp_project\notebooks,
# then project root is one level up.
current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
AUDIT_DIR = DATA_DIR / "audit"

TARGET_TENOR_DAYS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
TENORS = np.array(TARGET_TENOR_DAYS, dtype=float)
YEARS = TENORS / 365.0

RAW_V07_PARQUET = PROCESSED_DATA_DIR / "vix_term_structure_history.parquet"
RAW_V07_CSV = PROCESSED_DATA_DIR / "vix_term_structure_history.csv"

FINAL_REPAIRED_METHODOLOGY_VERSION = "v0.7.1_repaired_total_variance_cboe_anchors"

FINAL_REPAIRED_CSV = PROCESSED_DATA_DIR / "vix_term_structure_history_v0_7_1_repaired_total_variance.csv"
FINAL_REPAIRED_PARQUET = PROCESSED_DATA_DIR / "vix_term_structure_history_v0_7_1_repaired_total_variance.parquet"

FINAL_REPAIR_LOG = AUDIT_DIR / "v0_7_1_repaired_total_variance_repair_log.csv"
FINAL_CURVE_QA = AUDIT_DIR / "v0_7_1_repaired_total_variance_curve_qa.csv"
FINAL_DAILY_QA = AUDIT_DIR / "v0_7_1_repaired_total_variance_daily_qa.csv"
FINAL_COMBINED_QA = AUDIT_DIR / "v0_7_1_repaired_total_variance_combined_qa.csv"

CBOE_VIX_PATH = EXTERNAL_DATA_DIR / "cboe_vix_history.csv"
CBOE_VIX9D_PATH = EXTERNAL_DATA_DIR / "cboe_vix9d_history.csv"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Current directory:", current_dir)
print("Project root:", PROJECT_ROOT)
print("Processed data dir:", PROCESSED_DATA_DIR)
print("External data dir:", EXTERNAL_DATA_DIR)
print("Audit dir:", AUDIT_DIR)

print("\nRaw v0.7 parquet exists:", RAW_V07_PARQUET.exists())
print("Raw v0.7 CSV exists:", RAW_V07_CSV.exists())
print("Cboe VIX file exists:", CBOE_VIX_PATH.exists())
print("Cboe VIX9D file exists:", CBOE_VIX9D_PATH.exists())

## 2. Load raw v0.7 history and Cboe anchors

In [ ]:
def load_raw_v07_history():
    if RAW_V07_PARQUET.exists():
        return pd.read_parquet(RAW_V07_PARQUET).copy()

    if RAW_V07_CSV.exists():
        return pd.read_csv(RAW_V07_CSV).copy()

    raise FileNotFoundError(
        f"Could not find raw v0.7 history at {RAW_V07_PARQUET} or {RAW_V07_CSV}"
    )


def normalize_cboe_history(df, symbol):
    """
    Normalize Cboe history to columns:
      trade_date, vix_close / vix9d_close
    """
    symbol = symbol.upper()
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    if "trade_date" in df.columns:
        return df

    date_col = None
    for candidate in ["DATE", "Date", "date"]:
        if candidate in df.columns:
            date_col = candidate
            break

    close_col = None
    for candidate in ["CLOSE", "Close", "close"]:
        if candidate in df.columns:
            close_col = candidate
            break

    if date_col is None:
        raise ValueError(f"Could not find Cboe date column for {symbol}. Columns: {list(df.columns)}")

    if close_col is None:
        raise ValueError(f"Could not find Cboe close column for {symbol}. Columns: {list(df.columns)}")

    close_name = "vix_close" if symbol == "VIX" else "vix9d_close"

    df["trade_date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y%m%d").astype(int)
    df[close_name] = pd.to_numeric(df[close_col], errors="coerce")

    return (
        df[["trade_date", close_name]]
        .dropna()
        .drop_duplicates("trade_date")
        .sort_values("trade_date")
        .reset_index(drop=True)
    )


def load_cboe_history_file(local_path, symbol):
    if not local_path.exists():
        raise FileNotFoundError(
            f"Missing {symbol} file at {local_path}. "
            "Run notebook 02_compare_vix_benchmarks_v0_7.ipynb first, or place the Cboe CSV here."
        )

    return normalize_cboe_history(pd.read_csv(local_path), symbol)


raw_df = load_raw_v07_history()

cboe_vix_df = load_cboe_history_file(CBOE_VIX_PATH, "VIX")
cboe_vix9d_df = load_cboe_history_file(CBOE_VIX9D_PATH, "VIX9D")

cboe_anchor_df = (
    cboe_vix9d_df[["trade_date", "vix9d_close"]]
    .merge(cboe_vix_df[["trade_date", "vix_close"]], on="trade_date", how="outer")
)

print("Raw v0.7 rows:", len(raw_df))
print("Raw date range:", raw_df["trade_date"].min(), "to", raw_df["trade_date"].max())

print("\nRaw methodology versions:")
display(raw_df["methodology_version"].value_counts())

print("\nCboe VIX rows:", len(cboe_vix_df))
print("Cboe VIX9D rows:", len(cboe_vix9d_df))

display(raw_df.head())

## 3. Raw data completeness check

In [ ]:
counts = (
    raw_df
    .groupby("trade_date")
    .agg(
        rows=("target_days", "count"),
        unique_tenors=("target_days", "nunique"),
        min_tenor=("target_days", "min"),
        max_tenor=("target_days", "max"),
    )
    .reset_index()
)

bad_counts = counts[
    (counts["rows"] != len(TARGET_TENOR_DAYS))
    | (counts["unique_tenors"] != len(TARGET_TENOR_DAYS))
    | (counts["min_tenor"] != min(TARGET_TENOR_DAYS))
    | (counts["max_tenor"] != max(TARGET_TENOR_DAYS))
]

duplicates = (
    raw_df
    .groupby(["trade_date", "target_days"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicates[duplicates["count"] > 1]

print("Unique trade dates:", raw_df["trade_date"].nunique())
print("Expected rows:", raw_df["trade_date"].nunique() * len(TARGET_TENOR_DAYS))
print("Actual rows:", len(raw_df))

print("\nBad/incomplete dates:", len(bad_counts))
display(bad_counts)

print("\nDuplicate date/tenor rows:", len(duplicates))
display(duplicates)

## 4. Build raw curve matrices

In [ ]:
raw_vol_curve_df = (
    raw_df
    .pivot(index="trade_date", columns="target_days", values="vix_style_vol")
    .sort_index()
)

raw_var_curve_df = (
    raw_df
    .pivot(index="trade_date", columns="target_days", values="implied_variance")
    .sort_index()
)

print("Raw vol matrix shape:", raw_vol_curve_df.shape)
print("Raw variance matrix shape:", raw_var_curve_df.shape)

display(raw_vol_curve_df.head())
display(raw_vol_curve_df.tail())

## 5. Repair configuration

The repair list below was chosen after benchmark and curve-shape review. These dates had obvious raw curve-quality problems such as impossible valleys or negative total-variance forward segments.

Do not add dates just because they differ from Cboe. Add dates only when the internal curve shape is clearly broken.

In [ ]:
REPAIR_DATES = [
    20200228,
    20200312,
    20200318,
    20201218,
    20210129,
    20210225,
    20211126,
    20220120,
    20241218,
]

print("Repair dates:", REPAIR_DATES)
print("Repair rows expected:", len(REPAIR_DATES) * len(TARGET_TENOR_DAYS))

## 6. Repair helper functions

The repair is done in total variance space:

`total variance(T) = annual variance(T) × T`

The repaired curve starts from neighboring clean curves, anchors 9d to VIX9D and 30d to VIX when available, and enforces non-decreasing total variance.

In [ ]:
all_raw_dates = list(raw_var_curve_df.index)


def date_int_to_timestamp(date_int):
    return pd.to_datetime(str(int(date_int)), format="%Y%m%d")


def get_neighbor_dates_for_repair(trade_date, repair_dates):
    idx = all_raw_dates.index(trade_date)

    prev_date = None
    for j in range(idx - 1, -1, -1):
        candidate = all_raw_dates[j]
        if candidate not in repair_dates:
            prev_date = candidate
            break

    next_date = None
    for j in range(idx + 1, len(all_raw_dates)):
        candidate = all_raw_dates[j]
        if candidate not in repair_dates:
            next_date = candidate
            break

    if prev_date is None or next_date is None:
        raise ValueError(f"Could not find clean neighbors for {trade_date}")

    return prev_date, next_date


def interpolate_neighbor_total_variance(trade_date, repair_dates):
    prev_date, next_date = get_neighbor_dates_for_repair(trade_date, repair_dates)

    prev_ts = date_int_to_timestamp(prev_date)
    trade_ts = date_int_to_timestamp(trade_date)
    next_ts = date_int_to_timestamp(next_date)

    # Calendar-day interpolation between the neighboring clean curves.
    w = (trade_ts - prev_ts).days / (next_ts - prev_ts).days

    prev_ann_var = raw_var_curve_df.loc[prev_date, TARGET_TENOR_DAYS].astype(float).values
    next_ann_var = raw_var_curve_df.loc[next_date, TARGET_TENOR_DAYS].astype(float).values

    prev_total_var = prev_ann_var * YEARS
    next_total_var = next_ann_var * YEARS

    base_total_var = (1 - w) * prev_total_var + w * next_total_var

    return prev_date, next_date, base_total_var


def make_repaired_total_variance_curve(trade_date, repair_dates, min_forward_variance=0.0001):
    """
    Repair in total-variance space.

    Steps:
      1. Start from previous/next clean-date interpolated total variance curve.
      2. Anchor 9d to Cboe VIX9D and 30d to Cboe VIX when available.
      3. Preserve neighbor-curve curvature around the 9d/30d anchor line.
      4. Enforce non-decreasing total variance.
    """
    prev_date, next_date, base_tv = interpolate_neighbor_total_variance(
        trade_date=trade_date,
        repair_dates=repair_dates,
    )

    anchor_row = cboe_anchor_df[cboe_anchor_df["trade_date"] == trade_date]

    repaired_tv = base_tv.copy()
    anchor_used = "none"

    if len(anchor_row) == 1:
        anchor_row = anchor_row.iloc[0]

        has_9d_anchor = pd.notna(anchor_row.get("vix9d_close"))
        has_30d_anchor = pd.notna(anchor_row.get("vix_close"))

        if has_9d_anchor and has_30d_anchor:
            anchor_9_tv = (float(anchor_row["vix9d_close"]) / 100) ** 2 * (9 / 365)
            anchor_30_tv = (float(anchor_row["vix_close"]) / 100) ** 2 * (30 / 365)

            base_9_tv = base_tv[TARGET_TENOR_DAYS.index(9)]
            base_30_tv = base_tv[TARGET_TENOR_DAYS.index(30)]

            base_line = np.interp(
                TENORS,
                [9, 30],
                [base_9_tv, base_30_tv],
            )

            target_line = np.interp(
                TENORS,
                [9, 30],
                [anchor_9_tv, anchor_30_tv],
            )

            repaired_tv = target_line + (base_tv - base_line)

            # Force exact anchor values.
            repaired_tv[TARGET_TENOR_DAYS.index(9)] = anchor_9_tv
            repaired_tv[TARGET_TENOR_DAYS.index(30)] = anchor_30_tv

            # For 33d, preserve the base 30d->33d total variance increment,
            # but do not allow a negative forward segment.
            idx_30 = TARGET_TENOR_DAYS.index(30)
            idx_33 = TARGET_TENOR_DAYS.index(33)

            base_30_33_increment = base_tv[idx_33] - base_tv[idx_30]
            min_30_33_increment = min_forward_variance * ((33 - 30) / 365)

            repaired_tv[idx_33] = (
                repaired_tv[idx_30]
                + max(base_30_33_increment, min_30_33_increment)
            )

            anchor_used = "cboe_9d_30d"

        elif has_30d_anchor:
            anchor_30_tv = (float(anchor_row["vix_close"]) / 100) ** 2 * (30 / 365)
            idx_30 = TARGET_TENOR_DAYS.index(30)
            shift = anchor_30_tv - base_tv[idx_30]
            repaired_tv = base_tv + shift
            repaired_tv[idx_30] = anchor_30_tv
            anchor_used = "cboe_30d"

        elif has_9d_anchor:
            anchor_9_tv = (float(anchor_row["vix9d_close"]) / 100) ** 2 * (9 / 365)
            idx_9 = TARGET_TENOR_DAYS.index(9)
            shift = anchor_9_tv - base_tv[idx_9]
            repaired_tv = base_tv + shift
            repaired_tv[idx_9] = anchor_9_tv
            anchor_used = "cboe_9d"

    # Enforce non-decreasing total variance with a tiny positive forward variance floor.
    repaired_tv_monotone = repaired_tv.copy()

    for i in range(1, len(repaired_tv_monotone)):
        dt = YEARS[i] - YEARS[i - 1]
        floor_value = repaired_tv_monotone[i - 1] + min_forward_variance * dt

        if repaired_tv_monotone[i] < floor_value:
            repaired_tv_monotone[i] = floor_value

    repaired_ann_var = repaired_tv_monotone / YEARS
    repaired_ann_var = np.maximum(repaired_ann_var, 0.000001)

    return {
        "prev_date": prev_date,
        "next_date": next_date,
        "anchor_used": anchor_used,
        "repaired_ann_var": pd.Series(repaired_ann_var, index=TARGET_TENOR_DAYS),
        "repaired_total_var": pd.Series(repaired_tv_monotone, index=TARGET_TENOR_DAYS),
    }

## 7. Create candidate repaired dataset

In [ ]:
candidate_df = raw_df.copy()

candidate_df["raw_implied_variance"] = candidate_df["implied_variance"]
candidate_df["raw_vix_style_vol"] = candidate_df["vix_style_vol"]
candidate_df["is_repaired"] = False
candidate_df["repair_method"] = ""
candidate_df["source_methodology_version"] = candidate_df["methodology_version"]

repair_log_rows = []

for trade_date in REPAIR_DATES:
    repair_result = make_repaired_total_variance_curve(
        trade_date=trade_date,
        repair_dates=REPAIR_DATES,
    )

    repaired_ann_var = repair_result["repaired_ann_var"]

    for tenor in TARGET_TENOR_DAYS:
        mask = (
            (candidate_df["trade_date"] == trade_date)
            & (candidate_df["target_days"] == tenor)
        )

        if mask.sum() != 1:
            raise ValueError(f"Expected one row for {trade_date}, {tenor}; found {mask.sum()}")

        old_var = float(candidate_df.loc[mask, "implied_variance"].iloc[0])
        old_vol = float(candidate_df.loc[mask, "vix_style_vol"].iloc[0])

        new_var = float(repaired_ann_var.loc[tenor])
        new_vol = np.sqrt(new_var) * 100

        candidate_df.loc[mask, "implied_variance"] = new_var
        candidate_df.loc[mask, "vix_style_vol"] = new_vol
        candidate_df.loc[mask, "is_repaired"] = True
        candidate_df.loc[mask, "repair_method"] = (
            "neighbor_total_variance_interp_cboe_9d_30d_anchor_monotone"
        )

        repair_log_rows.append({
            "trade_date": trade_date,
            "target_days": tenor,
            "prev_neighbor_date": repair_result["prev_date"],
            "next_neighbor_date": repair_result["next_date"],
            "anchor_used": repair_result["anchor_used"],
            "old_implied_variance": old_var,
            "new_implied_variance": new_var,
            "old_vix_style_vol": old_vol,
            "new_vix_style_vol": new_vol,
            "vol_change": new_vol - old_vol,
        })

repair_log_df = pd.DataFrame(repair_log_rows)

candidate_vol_curve_df = (
    candidate_df
    .pivot(index="trade_date", columns="target_days", values="vix_style_vol")
    .sort_index()
)

candidate_var_curve_df = (
    candidate_df
    .pivot(index="trade_date", columns="target_days", values="implied_variance")
    .sort_index()
)

print("Candidate repaired rows:", int(candidate_df["is_repaired"].sum()))
print("Candidate repaired dates:", [int(x) for x in sorted(candidate_df.loc[candidate_df["is_repaired"], "trade_date"].unique())])

display(repair_log_df)

## 8. Inspect raw vs repaired curves

In [ ]:
print("Raw curves for repaired dates:")
display(raw_vol_curve_df.loc[REPAIR_DATES, TARGET_TENOR_DAYS])

print("Candidate repaired curves:")
display(candidate_vol_curve_df.loc[REPAIR_DATES, TARGET_TENOR_DAYS])

print("Change from repair:")
repair_change_df = (
    candidate_vol_curve_df.loc[REPAIR_DATES, TARGET_TENOR_DAYS]
    - raw_vol_curve_df.loc[REPAIR_DATES, TARGET_TENOR_DAYS]
)
display(repair_change_df)

## 9. Forward-variance check on repaired dates

This cell verifies that the repaired curves do not have negative total-variance forward segments.

In [ ]:
for focus_date in REPAIR_DATES:
    vols = candidate_vol_curve_df.loc[focus_date, TARGET_TENOR_DAYS].astype(float)
    ann_vars = candidate_var_curve_df.loc[focus_date, TARGET_TENOR_DAYS].astype(float)

    total_vars = ann_vars.values * YEARS
    forward_vars = np.diff(total_vars) / np.diff(YEARS)

    tv_df = pd.DataFrame({
        "target_days": TARGET_TENOR_DAYS,
        "vol": vols.values,
        "annual_variance": ann_vars.values,
        "total_variance": total_vars,
    })

    fv_df = pd.DataFrame({
        "from_tenor": TARGET_TENOR_DAYS[:-1],
        "to_tenor": TARGET_TENOR_DAYS[1:],
        "forward_variance": forward_vars,
        "forward_vol": np.sqrt(np.maximum(forward_vars, 0)) * 100,
    })

    print("\n" + "=" * 100)
    print("Date:", focus_date)
    display(tv_df)
    display(fv_df)

## 10. Local windows around repaired dates

In [ ]:
all_dates = sorted(candidate_vol_curve_df.index)

for focus_date in REPAIR_DATES:
    idx = all_dates.index(focus_date)
    window_dates = all_dates[max(0, idx - 3): idx + 4]

    print("\n" + "=" * 100)
    print("Raw v0.7 window around:", focus_date)
    display(raw_vol_curve_df.loc[window_dates, TARGET_TENOR_DAYS])

    print("Candidate repaired window around:", focus_date)
    display(candidate_vol_curve_df.loc[window_dates, TARGET_TENOR_DAYS])

## 11. Plot local windows around repaired dates

In [ ]:
for focus_date in REPAIR_DATES:
    idx = all_dates.index(focus_date)
    window_dates = all_dates[max(0, idx - 3): idx + 4]

    plot_df = candidate_vol_curve_df.loc[window_dates, TARGET_TENOR_DAYS].copy()

    plt.figure(figsize=(10, 5))

    for date in plot_df.index:
        label = str(date)
        linewidth = 3 if date == focus_date else 1
        alpha = 1.0 if date == focus_date else 0.6

        plt.plot(
            TARGET_TENOR_DAYS,
            plot_df.loc[date, TARGET_TENOR_DAYS],
            marker="o",
            label=label,
            linewidth=linewidth,
            alpha=alpha,
        )

    plt.title(f"Repaired term structure window around {focus_date}")
    plt.xlabel("Target tenor days")
    plt.ylabel("VIX-style vol")
    plt.legend()
    plt.show()

## 12. QA helper functions

In [ ]:
def calculate_curve_shape_qa(vol_curve_df, var_curve_df):
    qa_rows = []

    for trade_date in vol_curve_df.index:
        vols = vol_curve_df.loc[trade_date, TARGET_TENOR_DAYS].astype(float)
        variances = var_curve_df.loc[trade_date, TARGET_TENOR_DAYS].astype(float)

        adjacent_vol_diffs = vols.diff().dropna()
        max_adjacent_abs_move = adjacent_vol_diffs.abs().max()
        worst_adjacent_to_tenor = adjacent_vol_diffs.abs().idxmax()

        max_local_kink = 0.0
        worst_kink_tenor = None

        for i in range(1, len(TARGET_TENOR_DAYS) - 1):
            left_tenor = TARGET_TENOR_DAYS[i - 1]
            mid_tenor = TARGET_TENOR_DAYS[i]
            right_tenor = TARGET_TENOR_DAYS[i + 1]

            expected_mid = 0.5 * (vols[left_tenor] + vols[right_tenor])
            kink = vols[mid_tenor] - expected_mid

            if abs(kink) > abs(max_local_kink):
                max_local_kink = kink
                worst_kink_tenor = mid_tenor

        total_variance = variances.values * YEARS
        forward_variance = np.diff(total_variance) / np.diff(YEARS)

        qa_rows.append({
            "trade_date": trade_date,
            "min_vol": vols.min(),
            "max_vol": vols.max(),
            "vol_range": vols.max() - vols.min(),
            "max_adjacent_abs_move": max_adjacent_abs_move,
            "worst_adjacent_to_tenor": worst_adjacent_to_tenor,
            "max_local_kink": max_local_kink,
            "max_local_kink_abs": abs(max_local_kink),
            "worst_kink_tenor": worst_kink_tenor,
            "min_forward_variance": np.min(forward_variance),
            "max_forward_variance": np.max(forward_variance),
        })

    curve_qa_df = pd.DataFrame(qa_rows)

    curve_qa_df["flag_large_kink"] = curve_qa_df["max_local_kink_abs"] >= 3.0
    curve_qa_df["flag_large_adjacent_move"] = curve_qa_df["max_adjacent_abs_move"] >= 5.0
    curve_qa_df["flag_negative_forward_variance"] = curve_qa_df["min_forward_variance"] < -0.005

    curve_qa_df["curve_shape_flag"] = (
        curve_qa_df["flag_large_kink"]
        | curve_qa_df["flag_large_adjacent_move"]
        | curve_qa_df["flag_negative_forward_variance"]
    )

    return curve_qa_df


def calculate_daily_change_qa(vol_curve_df):
    daily_vol_change_df = vol_curve_df.diff()
    daily_qa_rows = []

    for trade_date in daily_vol_change_df.index[1:]:
        changes = daily_vol_change_df.loc[trade_date, TARGET_TENOR_DAYS].astype(float)

        median_change = changes.median()
        residual_changes = changes - median_change

        max_abs_daily_move = changes.abs().max()
        worst_move_tenor = changes.abs().idxmax()

        max_abs_residual_move = residual_changes.abs().max()
        worst_residual_tenor = residual_changes.abs().idxmax()

        daily_qa_rows.append({
            "trade_date": trade_date,
            "median_curve_change": median_change,
            "max_abs_daily_move": max_abs_daily_move,
            "worst_move_tenor": worst_move_tenor,
            "max_abs_residual_move": max_abs_residual_move,
            "worst_residual_tenor": worst_residual_tenor,
        })

    daily_qa_df = pd.DataFrame(daily_qa_rows)

    daily_qa_df["flag_large_daily_move"] = daily_qa_df["max_abs_daily_move"] >= 15.0
    daily_qa_df["flag_single_tenor_glitch"] = daily_qa_df["max_abs_residual_move"] >= 5.0

    daily_qa_df["daily_change_flag"] = (
        daily_qa_df["flag_large_daily_move"]
        | daily_qa_df["flag_single_tenor_glitch"]
    )

    return daily_qa_df


def combine_qa(curve_qa_df, daily_qa_df, repaired_dates):
    combined_qa_df = (
        curve_qa_df
        .merge(
            daily_qa_df[
                [
                    "trade_date",
                    "median_curve_change",
                    "max_abs_daily_move",
                    "worst_move_tenor",
                    "max_abs_residual_move",
                    "worst_residual_tenor",
                    "daily_change_flag",
                ]
            ],
            on="trade_date",
            how="left",
        )
    )

    combined_qa_df["daily_change_flag"] = (
        combined_qa_df["daily_change_flag"]
        .fillna(False)
        .astype(bool)
    )

    combined_qa_df["is_repaired_date"] = (
        combined_qa_df["trade_date"].isin(repaired_dates)
    )

    combined_qa_df["high_priority_flag"] = (
        combined_qa_df["curve_shape_flag"]
        | (
            combined_qa_df["daily_change_flag"]
            & (combined_qa_df["max_abs_residual_move"] >= 8.0)
        )
        | combined_qa_df["is_repaired_date"]
    )

    return combined_qa_df

## 13. Broad QA on repaired dataset

In [ ]:
curve_qa_df = calculate_curve_shape_qa(
    vol_curve_df=candidate_vol_curve_df,
    var_curve_df=candidate_var_curve_df,
)

daily_qa_df = calculate_daily_change_qa(
    vol_curve_df=candidate_vol_curve_df,
)

repaired_dates = (
    candidate_df[candidate_df["is_repaired"] == True]["trade_date"]
    .drop_duplicates()
    .tolist()
)

combined_qa_df = combine_qa(
    curve_qa_df=curve_qa_df,
    daily_qa_df=daily_qa_df,
    repaired_dates=repaired_dates,
)

print("Curve-shape flagged dates:", int(curve_qa_df["curve_shape_flag"].sum()))
print("Negative-forward-variance dates:", int(curve_qa_df["flag_negative_forward_variance"].sum()))
print("Daily-change flagged dates:", int(daily_qa_df["daily_change_flag"].sum()))
print("High-priority QA dates:", int(combined_qa_df["high_priority_flag"].sum()))

print("\nCurve-shape flags:")
display(
    curve_qa_df[curve_qa_df["curve_shape_flag"]]
    .sort_values(
        ["flag_negative_forward_variance", "max_local_kink_abs", "max_adjacent_abs_move"],
        ascending=False,
    )
    .head(75)
)

print("\nDaily-change flags:")
display(
    daily_qa_df[daily_qa_df["daily_change_flag"]]
    .sort_values(
        ["flag_single_tenor_glitch", "max_abs_residual_move", "max_abs_daily_move"],
        ascending=False,
    )
    .head(75)
)

print("\nHigh-priority QA review:")
display(
    combined_qa_df[combined_qa_df["high_priority_flag"]]
    .sort_values(
        [
            "curve_shape_flag",
            "is_repaired_date",
            "max_local_kink_abs",
            "max_abs_residual_move",
        ],
        ascending=False,
    )
    .head(100)
)

## 14. Final validation before save

In [ ]:
final_repaired_df = candidate_df.copy()

# Set all rows to the final repaired methodology version.
final_repaired_df["methodology_version"] = FINAL_REPAIRED_METHODOLOGY_VERSION

# Keep the raw methodology version for audit.
if "source_methodology_version" not in final_repaired_df.columns:
    final_repaired_df["source_methodology_version"] = "v0.7_exchange_calendar_fred_sofr_market_close"

counts_final = (
    final_repaired_df
    .groupby("trade_date")
    .agg(
        rows=("target_days", "count"),
        unique_tenors=("target_days", "nunique"),
        min_tenor=("target_days", "min"),
        max_tenor=("target_days", "max"),
    )
    .reset_index()
)

bad_counts_final = counts_final[
    (counts_final["rows"] != len(TARGET_TENOR_DAYS))
    | (counts_final["unique_tenors"] != len(TARGET_TENOR_DAYS))
    | (counts_final["min_tenor"] != min(TARGET_TENOR_DAYS))
    | (counts_final["max_tenor"] != max(TARGET_TENOR_DAYS))
]

duplicate_final = (
    final_repaired_df
    .groupby(["trade_date", "target_days"])
    .size()
    .reset_index(name="count")
)
duplicate_final = duplicate_final[duplicate_final["count"] > 1]

print("Rows:", len(final_repaired_df))
print("Unique trade dates:", final_repaired_df["trade_date"].nunique())
print("Expected rows:", final_repaired_df["trade_date"].nunique() * len(TARGET_TENOR_DAYS))
print("Repaired rows:", int(final_repaired_df["is_repaired"].sum()))
print("Repaired dates:", [int(x) for x in sorted(final_repaired_df.loc[final_repaired_df["is_repaired"], "trade_date"].unique())])

print("\nBad/incomplete dates:", len(bad_counts_final))
display(bad_counts_final)

print("\nDuplicate date/tenor rows:", len(duplicate_final))
display(duplicate_final)

print("\nMethodology versions:")
display(final_repaired_df["methodology_version"].value_counts())

print("\nSource methodology versions:")
display(final_repaired_df["source_methodology_version"].value_counts())

print("\nNegative-forward-variance QA dates after repair:")
display(curve_qa_df[curve_qa_df["flag_negative_forward_variance"]])

## 15. Save final repaired dataset and audit files

Run this cell only after the QA output above looks acceptable.

In [ ]:
final_repaired_df.to_csv(FINAL_REPAIRED_CSV, index=False)
final_repaired_df.to_parquet(FINAL_REPAIRED_PARQUET, index=False)

repair_log_df.to_csv(FINAL_REPAIR_LOG, index=False)
curve_qa_df.to_csv(FINAL_CURVE_QA, index=False)
daily_qa_df.to_csv(FINAL_DAILY_QA, index=False)
combined_qa_df.to_csv(FINAL_COMBINED_QA, index=False)

print("Saved final repaired CSV:", FINAL_REPAIRED_CSV)
print("Saved final repaired parquet:", FINAL_REPAIRED_PARQUET)
print("Saved repair log:", FINAL_REPAIR_LOG)
print("Saved curve QA:", FINAL_CURVE_QA)
print("Saved daily QA:", FINAL_DAILY_QA)
print("Saved combined QA:", FINAL_COMBINED_QA)

print("\nRows:", len(final_repaired_df))
print("Repaired rows:", int(final_repaired_df["is_repaired"].sum()))
print("Repaired dates:", [int(x) for x in sorted(final_repaired_df.loc[final_repaired_df["is_repaired"], "trade_date"].unique())])

## 16. Optional: reload saved file and confirm

This is a final smoke test that the saved parquet can be read back correctly.

In [ ]:
saved_check_df = pd.read_parquet(FINAL_REPAIRED_PARQUET)

print("Saved file rows:", len(saved_check_df))
print("Saved file date range:", saved_check_df["trade_date"].min(), "to", saved_check_df["trade_date"].max())

print("\nSaved methodology versions:")
display(saved_check_df["methodology_version"].value_counts())

print("\nSaved repaired row count:")
display(saved_check_df["is_repaired"].value_counts(dropna=False))